# Ingestão e descompactação dos dados brutos

Este notebook realiza a ingestão e descompactação do conjunto de dados bruto em passos simples:

1. Configuração de diretórios  
    - Cria/garante os diretórios de trabalho: DATA_DIR (`data`), RAW_DIR (`data/raw`), STAGED_DIR (`data/staged`) e PROCESSED_DIR (`data/processed`) via `ensure_dirs()`.  
    - Logs são emitidos pelo `logger` para acompanhar o processo.

2. Download do arquivo bruto (.zip)  
    - `download_dataset(url, out_path)` baixa o arquivo a partir da URL fornecida e salva em `out_path` (`data/raw/dados-metereologicos-2018-2024-inmet.zip`) usando requisições em streaming para não carregar tudo na memória.

3. Validações básicas do arquivo  
    - `check_file_exists(out_path)` verifica se o arquivo foi efetivamente baixado.  
    - `check_file_type(out_path, ".zip")` confirma que o arquivo é um .zip.

4. Descompactação  
    - `unzip_file(out_path, RAW_DIR)` extrai o conteúdo do .zip para `RAW_DIR` (`data/raw`).

5. Verificação pós-descompactação  
    - `check_all_files_raw()` lista e valida a presença dos arquivos descompactados em `RAW_DIR`.

Observações:
- Variáveis úteis definidas no notebook: `CHUNK_SIZE` (100000) para processamento em lotes, caminhos `DATA_DIR`, `RAW_DIR`, `STAGED_DIR`, `PROCESSED_DIR`, além de `url` e `out_path`.  
- Logs informativos acompanham cada etapa para facilitar depuração e auditoria.

In [2]:
import os
import logging
from pathlib import Path
import zipfile
import glob
import requests

In [3]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
# Config
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = DATA_DIR / "raw"
STAGED_DIR = DATA_DIR / "staged"
PROCESSED_DIR = DATA_DIR / "processed"

# Defaults
CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", "100000"))

In [ ]:
def ensure_dirs():
    logger.info("Ensuring data directories exist.")
    if not RAW_DIR.exists():
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created RAW_DIR at {RAW_DIR}")
    if not STAGED_DIR.exists():
        STAGED_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created STAGED_DIR at {STAGED_DIR}")
    if not PROCESSED_DIR.exists():
        PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created PROCESSED_DIR at {PROCESSED_DIR}")

In [ ]:
url = "https://www.kaggle.com/api/v1/datasets/download/gnomows/dados-metereologicos-2018-2024-inmet"
out_path = RAW_DIR / "dados-metereologicos-2018-2024-inmet.zip"

def download_dataset(url, out_path):
    logger.info(f"Downloading dataset from {url} to {out_path}")
    with requests.get(url, stream=True, allow_redirects=True) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    logger.info(f"Downloaded to {out_path}")

In [7]:
ensure_dirs()
download_dataset(url, out_path)


INFO:__main__:Ensuring data directories exist.
INFO:__main__:Created RAW_DIR at data/raw
INFO:__main__:Created STAGED_DIR at data/staged
INFO:__main__:Created PROCESSED_DIR at data/processed
INFO:__main__:Downloading dataset from https://www.kaggle.com/api/v1/datasets/download/gnomows/dados-metereologicos-2018-2024-inmet to data/raw/dados-metereologicos-2018-2024-inmet.zip


Downloaded to data/raw/dados-metereologicos-2018-2024-inmet.zip


In [8]:
def check_file_exists(file_path):
    if not file_path.exists():
        logger.error(f"File {file_path} does not exist.")
        raise FileNotFoundError(f"File {file_path} does not exist.")
    logger.info(f"File {file_path} exists.")
    return True

def check_file_type(file_path, expected_extension):
    if file_path.suffix != expected_extension:
        logger.error(f"File {file_path} is not of type {expected_extension}.")
        raise ValueError(f"File {file_path} is not of type {expected_extension}.")
    logger.info(f"File {file_path} is of type {expected_extension}.")
    return True

In [9]:
check_file_exists(out_path)
check_file_type(out_path, ".zip")

INFO:__main__:File data/raw/dados-metereologicos-2018-2024-inmet.zip exists.
INFO:__main__:File data/raw/dados-metereologicos-2018-2024-inmet.zip is of type .zip.


True

In [ ]:
def unzip_file(zip_path, extract_to):
    if not extract_to.exists():
        extract_to.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created extraction directory at {extract_to}")
    logger.info(f"Unzipping file {zip_path} to {extract_to}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    logger.info(f"Unzipped file to {extract_to}")

In [11]:
unzip_file(out_path, RAW_DIR)

INFO:__main__:Unzipping file data/raw/dados-metereologicos-2018-2024-inmet.zip to data/raw
INFO:__main__:Unzipped file to data/raw


In [12]:
def check_all_files_raw():
    files = list(RAW_DIR.glob("*"))
    if not files:
        logger.error(f"No files found in {RAW_DIR}")
        raise FileNotFoundError(f"No files found in {RAW_DIR}")
    for file in files:
        logger.info(f"Found file: {file}")
    return files

In [13]:
check_all_files_raw()

INFO:__main__:Found file: data/raw/dados-metereologicos-2018-2024-inmet.zip
INFO:__main__:Found file: data/raw/2021.csv
INFO:__main__:Found file: data/raw/2019.csv
INFO:__main__:Found file: data/raw/2020.csv
INFO:__main__:Found file: data/raw/2023.csv
INFO:__main__:Found file: data/raw/2018.csv
INFO:__main__:Found file: data/raw/2024.csv
INFO:__main__:Found file: data/raw/2022.csv


[PosixPath('data/raw/dados-metereologicos-2018-2024-inmet.zip'),
 PosixPath('data/raw/2021.csv'),
 PosixPath('data/raw/2019.csv'),
 PosixPath('data/raw/2020.csv'),
 PosixPath('data/raw/2023.csv'),
 PosixPath('data/raw/2018.csv'),
 PosixPath('data/raw/2024.csv'),
 PosixPath('data/raw/2022.csv')]